In [1]:
import torch
import torch.nn as nn
torch.set_default_dtype(torch.float64)
from collections import OrderedDict
import math
import numpy as np
import sympy as sp
import seaborn as sns
from scipy.interpolate import griddata
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy.stats import qmc
import scipy.io

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class Sin(torch.nn.Module):
    """sin 激活: y = sin(omega * x)"""
    def __init__(self, omega=1.0):
        super(Sin, self).__init__()
        self.omega = omega

    def forward(self, x):
        return torch.sin(self.omega * x)


class ChebyKANLayer(nn.Module):
    def __init__(self, input_dim, output_dim, degree):
        super(ChebyKANLayer, self).__init__()
        self.inputdim = input_dim
        self.outdim = output_dim
        self.degree = degree

        self.cheby_coeffs = nn.Parameter(torch.empty(input_dim, output_dim, degree + 1))
        nn.init.xavier_normal_(self.cheby_coeffs)
        self.register_buffer("arange", torch.arange(0, degree + 1, 1))    
        self.activation = Sin(omega=1.0)  # sin 激活 (消融: tanh -> sin, omega=1, 与 DNN 范本一致)

    def forward(self, x):
        x = torch.reshape(x, (-1, self.inputdim))
        x = self.activation(x)

        cheby = torch.ones(x.shape[0], self.inputdim, self.degree + 1, 
                          device=x.device)
        
        if self.degree > 0:
            cheby[:, :, 1] = x
        
        for i in range(2, self.degree + 1):
            cheby[:, :, i] = 2 * x * cheby[:, :, i - 1].clone() - cheby[:, :, i - 2].clone()

        y = torch.einsum('bid,iod->bo', cheby, self.cheby_coeffs)  
        y = y.view(-1, self.outdim)
        return y

# 修改DNNLayer，不再需要hidden_dim参数
class DNNLayer(nn.Module):
    def __init__(self, input_dim, output_dim, activation=Sin):
        super(DNNLayer, self).__init__()
        # 直接使用输入维度作为内部隐藏层维度（从layer_sizes推导）
        self.layer = nn.Sequential(
            nn.Linear(input_dim, input_dim),  # 隐藏层维度 = 输入维度
            activation(),
            nn.Linear(input_dim, output_dim)
        )
        
    def forward(self, x):
        return self.layer(x)

# 混合模型：仅使用layer_sizes定义所有维度
class HybridKAN_DNN(nn.Module):
    def __init__(self, layer_sizes, cheby_degree, lb, ub):
        super(HybridKAN_DNN, self).__init__()
        self.lb = lb
        self.ub = ub
        self.layers = nn.ModuleList()
        
        # 完全通过layer_sizes定义各层维度
        for i in range(len(layer_sizes) - 1):
            if i % 2 != 0:  # 偶数索引：ChebyKAN层
                self.layers.append(ChebyKANLayer(
                    input_dim=layer_sizes[i],
                    output_dim=layer_sizes[i+1],
                    degree=cheby_degree
                ))
            else:  # 奇数索引：DNN层（内部维度自动从layer_sizes[i]获取）
                self.layers.append(DNNLayer(
                    input_dim=layer_sizes[i],
                    output_dim=layer_sizes[i+1]
                ))
    
    def forward(self, x):
        out = 2 * (x - self.lb) / (self.ub - self.lb) - 1
        for layer in self.layers:
            out = layer(out)
        return out

In [4]:
# the physics-guided neural network
class PIDKN(nn.Module):
    def __init__(self, X_ub, X_lb, X_ic, X_res, q_ic, r_ic, layers, degree, lb, ub):
        super(PIDKN, self).__init__()
        # boundary conditions
        
        self.z_ub = torch.tensor(X_ub[:, 0:1], requires_grad=True).double().to(device)
        self.t_ub = torch.tensor(X_ub[:, 1:2], requires_grad=True).double().to(device)

        self.z_lb = torch.tensor(X_lb[:, 0:1], requires_grad=True).double().to(device)
        self.t_lb = torch.tensor(X_lb[:, 1:2], requires_grad=True).double().to(device)

        self.z_ic = torch.tensor(X_ic[:, 0:1], requires_grad=True).double().to(device)
        self.t_ic = torch.tensor(X_ic[:, 1:2], requires_grad=True).double().to(device)
        self.q_ic = torch.tensor(q_ic).double().to(device)
        self.r_ic = torch.tensor(r_ic).double().to(device)

        self.z_res = torch.tensor(X_res[:, 0:1], requires_grad=True).double().to(device)
        self.t_res = torch.tensor(X_res[:, 1:2], requires_grad=True).double().to(device)

        self.lb = torch.tensor(lb).double().to(device)  # 归一化下界
        self.ub = torch.tensor(ub).double().to(device)  # 归一

        self.layers = layers
        
        self.net = HybridKAN_DNN(layers, degree, self.lb, self.ub).to(device)

        self.iter = 1

        self.optimizer1 = torch.optim.Adam(self.net.parameters(),lr=1e-3)

        self.optimizer2 = torch.optim.LBFGS(
            self.net.parameters(), 
            lr=1.0, #步长缩放因子，用于控制拟牛顿算法中搜索的步长
            max_iter=30000, #最大迭代次数
            max_eval=30000, #函数和梯度最大评估次数
            history_size=50, #存储和更新历史信息的缓存大小
            tolerance_grad=1e-7, #表示梯度的容差，即梯度的变化小于该容差时，算法将终止
            tolerance_change=1.0 * np.finfo(float).eps, #表示优化变化的容差，即参数变化量的绝对值小于该容差时，算法将终止
            line_search_fn="strong_wolfe" #can be "strong_wolfe"
        )
        
        self.loss_history = {
            'total': [],
            'residual': [],
            'ic': [],
            'b': []
        }

    def net_h(self, z, t):  
        H = self.net(torch.cat([z, t], dim=1))
        q = H[:, 0:1]
        r = H[:, 1:2]
        return q, r
        
    def gradient(self, y, x):
        return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), retain_graph=True, create_graph=True)[0]
    
    def net_b(self, z, t):

        q, r = self.net_h(z, t)
        q_z = self.gradient(q, z)
        r_z = self.gradient(r, z)

        return q, r, q_z, r_z
                   
    
    def net_f(self, z, t):

        q, r = self.net_h(z, t)
        q_t = self.gradient(q, t)
        q_z = self.gradient(q, z)

        r_t = self.gradient(r, t)
        r_z = self.gradient(r, z)
        r_zz = self.gradient(r_z, z)
        r_zzz = self.gradient(r_zz, z)

        f1 = r_t - 4 * q_z - 6 * r * r_z
        # 8.29: 非线性系数改回 (2, 4), 与 ADM.py 递推和谱方法参考数据一致
        f2 = q_t - r_zzz - 2 * r * q_z - 4 * q * r_z

        return f1, f2

    def loss_func(self):

        self.optimizer1.zero_grad()
        self.optimizer2.zero_grad()

        q_ic_pred, r_ic_pred = self.net_h(self.z_ic, self.t_ic)

        q_ub_pred, r_ub_pred, q_z_ub_pred, r_z_ub_pred = self.net_b(self.z_ub, self.t_ub)

        q_lb_pred, r_lb_pred, q_z_lb_pred, r_z_lb_pred = self.net_b(self.z_lb, self.t_lb)

        f1_res_pred, f2_res_pred = self.net_f(self.z_res, self.t_res)

        loss_res1 = torch.mean(f1_res_pred**2)
        loss_res2 = torch.mean(f2_res_pred**2)
        loss_icq = torch.mean((q_ic_pred - self.q_ic)**2)
        loss_icr = torch.mean((r_ic_pred - self.r_ic)**2)

        # 周期边界: 两端的值与导数对应相等 (与谱方法格式一致)
        loss_b = (torch.mean((r_ub_pred - r_lb_pred)**2)
                + torch.mean((q_ub_pred - q_lb_pred)**2)
                + torch.mean((r_z_ub_pred - r_z_lb_pred)**2)
                + torch.mean((q_z_ub_pred - q_z_lb_pred)**2))

        loss_ic = loss_icq + loss_icr
        loss_res = loss_res1 + loss_res2

        loss = loss_res + loss_ic + loss_b
        
        loss.backward()
        self.iter += 1
        if self.iter % 100 == 0:
            self.loss_history['total'].append(loss.item())
            self.loss_history['residual'].append(loss_res.item())
            self.loss_history['ic'].append(loss_ic.item())
            self.loss_history['b'].append(loss_b.item())
            print(
                'Iter %d, Loss: %.5e, Loss_res: %.5e, Loss_ic: %.5e, Loss_b: %.5e' % 
                (   self.iter, 
                    loss.item(), 
                    loss_res.item(), 
                    loss_ic.item(),
                    loss_b.item()
                )
            )

        return loss
    
    # ---- 断点续训: checkpoint 保存/恢复 ----
    CKPT_PATHS = ('/kaggle/working/ckpt_dnnkan.pth',   # 本会话读写
                  '/kaggle/input/datasets/soliton4/inputs1/ckpt_dnnkan.pth')   # 上个会话的输出(Add Input挂载)

    def save_ckpt(self, phase, adam_done, lbfgs_done):
        import os
        path = self.CKPT_PATHS[0]
        data = {
            'phase': phase,
            'adam_done': adam_done,
            'lbfgs_done': lbfgs_done,
            'net': self.net.state_dict(),
            'opt1': self.optimizer1.state_dict(),
            'opt2': self.optimizer2.state_dict(),   # 含 L-BFGS 拟牛顿历史
            'loss_history': self.loss_history,
            'iter': self.iter,
        }
        tmp = path + '.tmp'
        torch.save(data, tmp)
        os.replace(tmp, path)   # 原子替换, 避免保存中途被杀产生损坏文件
        print(f'[ckpt] saved -> {path} (phase={phase}, adam={adam_done}, lbfgs={lbfgs_done})')

    def load_ckpt(self):
        import glob
        for pattern in self.CKPT_PATHS:
            for p in sorted(glob.glob(pattern)):
                try:
                    data = torch.load(p, map_location=device,weights_only=False)
                except Exception as e:
                    print(f'[ckpt] 跳过损坏的checkpoint {p}: {e}')
                    continue
                self.net.load_state_dict(data['net'])
                self.optimizer1.load_state_dict(data['opt1'])
                self.optimizer2.load_state_dict(data['opt2'])
                self.loss_history = data['loss_history']
                self.iter = data['iter']
                print(f'[ckpt] restored <- {p} (adam={data["adam_done"]}, lbfgs={data["lbfgs_done"]})')
                return data
        print('[ckpt] 未找到checkpoint, 从头训练')
        return None

    def _reset_lbfgs_count(self):
        # 分块调用 L-BFGS 时清零迭代计数; 拟牛顿历史(s/y)保留在 state 中不受影响
        for p in self.optimizer2.param_groups[0]['params']:
            st = self.optimizer2.state.get(p)
            if st:
                st['n_iter'] = 0
                st['func_evals'] = 0

    def train(self, adam_total=10000, lbfgs_total=30000,
              adam_every=1000, lbfgs_chunk=500):
        """分块训练并定期存 checkpoint; 启动时自动从 CKPT_PATHS 恢复续跑"""
        self.net.train()
        data = self.load_ckpt()
        adam_done = data['adam_done'] if data else 0
        lbfgs_done = data['lbfgs_done'] if data else 0

        if adam_done < adam_total:
            print("采用Adam优化器")
            for i in range(adam_done, adam_total):
                self.optimizer1.step(self.loss_func)
                if (i + 1) % adam_every == 0:
                    self.save_ckpt('adam', i + 1, 0)
            adam_done = adam_total
        else:
            print("Adam 阶段已完成, 跳过")

        if lbfgs_done < lbfgs_total:
            print("采用L-BFGS优化器")
            group = self.optimizer2.param_groups[0]
            group['max_iter'] = lbfgs_chunk      # 每块迭代数
            group['max_eval'] = 2 * lbfgs_chunk  # 给 line search 留出额外评估
            prev = None
            while lbfgs_done < lbfgs_total:
                self._reset_lbfgs_count()
                self.optimizer2.step(self.loss_func)
                st = self.optimizer2.state[group['params'][0]]
                n = st.get('n_iter', 0)
                if n == 0:
                    print('L-BFGS 满足收敛容差, 提前结束')
                    break
                lbfgs_done += n
                self.save_ckpt('lbfgs', adam_done, lbfgs_done)
                cur = self.loss_history['total'][-1] if self.loss_history['total'] else None
                if prev is not None and cur is not None and                         abs(prev - cur) < 1e-12 * (abs(prev) + abs(cur)):
                    print('L-BFGS 损失不再下降, 提前结束')
                    break
                prev = cur
        else:
            print("L-BFGS 阶段已完成, 跳过")

        self.save_ckpt('done', adam_done, lbfgs_done) 
            
    def predict(self, X):
        
        z = torch.tensor(X[:, 0:1], requires_grad=True).double().to(device)
        t = torch.tensor(X[:, 1:2], requires_grad=True).double().to(device)

        self.net.eval()
        q, r = self.net_h(z, t)
        q = q.detach().cpu().numpy()
        r = r.detach().cpu().numpy()

        return q, r

In [ ]:
N_res = 10000
N_b = 400
N_ic = 1000

x = np.linspace(-15, 15, 601)      # 8.29: 计算域扩到 [-15,15]
t = np.linspace(0, 0.3, 201)       

x_mesh, t_mesh = np.meshgrid(x, t)

X = np.hstack((x_mesh.flatten()[:,None], t_mesh.flatten()[:,None]))

lb = X.min(0) #下界x'QxQ
ub = X.max(0)  #上界 

# df_h = np.load("/kaggle/input/2w-400-1000/sampling_points-2W-400-1000.npz")
# X_res = df_h['X_res']
# X_ub = df_h['X_ub']
# X_lb = df_h['X_lb']
# X_ic = df_h['X_ic']

def lhs(n, size, seed):
    """生成n维拉丁超立方采样，返回size个样本点
    (固定seed: 会话中断续跑时重新执行本单元, 采样点与上次完全一致)"""
    sampler = qmc.LatinHypercube(d=n, seed=seed)
    return sampler.random(n=size)

# 使用简洁形式进行采样（与你要求的形式一致）
X_res = lb + (ub - lb) * lhs(2, N_res, seed=0)  # 2维残差点采样

# 边界点和初始条件点也采用类似风格的简洁写法
X_ub = np.hstack((ub[0] * np.ones((N_b, 1)), 
                  lb[1] + (ub[1] - lb[1]) * lhs(1, N_b, seed=1)))  # 上边界点

X_lb = np.hstack((lb[0] * np.ones((N_b, 1)), 
                  lb[1] + (ub[1] - lb[1]) * lhs(1, N_b, seed=2)))  # 下边界点

X_ic = np.hstack((lb[0] + (ub[0] - lb[0]) * lhs(1, N_ic, seed=3), 
                  lb[1] * np.ones((N_ic, 1))))  # 初始条件点

In [6]:
np.savez(
    "/kaggle/working/sampling_points-10000-400-1000.npz",  # 输出文件名称（后缀.npz可省略，函数会自动补充）
    X_res=X_res,            # 残差点采样数据
    X_ub=X_ub,              # 上边界点采样数据
    X_lb=X_lb,              # 下边界点采样数据
    X_ic=X_ic               # 初始条件点采样数据
)

print("采样点已成功存储到 sampling_points.npz 文件中")

采样点已成功存储到 sampling_points.npz 文件中


In [7]:
def funic(X):
    x = X[:, 0:1]
    # 8.29: 初始条件改为摊平高斯包 + q0 = 0
    # (与谱方法参考数据一致; 纯高斯与带限版差异 <2e-4, PINN 中直接用纯高斯即可)
    r = 0.3 * np.exp(-0.25 * x**2)
    q = np.zeros_like(x)
    return r, q

r_ic, q_ic = funic(X_ic)

In [8]:
degree = 4
layers = [2,80,22,80,22,80,22,2]

model = PIDKN(X_ub, X_lb, X_ic, X_res, q_ic, r_ic, layers, degree, lb, ub)

In [9]:
def count_parameters(model):
    """计算模型中可训练参数的总个数"""
    total_params = 0
    for param in model.parameters():
        # 累加每个参数张量的元素数量
        total_params += param.numel()
    return total_params

param_count = count_parameters(model)
print(f"模型可训练参数总个数: {param_count:,}")  # 使用逗号作为千位分隔符

模型可训练参数总个数: 31,890


In [10]:
model.train()

[ckpt] restored <- /kaggle/input/datasets/soliton4/inputs1/ckpt_dnnkan.pth (adam=10000, lbfgs=15000)
Adam 阶段已完成, 跳过
采用L-BFGS优化器
Iter 38700, Loss: 1.00518e-06, Loss_res: 2.16334e-07, Loss_ic: 7.85481e-07, Loss_b: 3.36289e-09
Iter 38800, Loss: 1.00515e-06, Loss_res: 2.16319e-07, Loss_ic: 7.85469e-07, Loss_b: 3.36228e-09
Iter 38900, Loss: 1.00512e-06, Loss_res: 2.16304e-07, Loss_ic: 7.85458e-07, Loss_b: 3.36167e-09
Iter 39000, Loss: 1.00510e-06, Loss_res: 2.16289e-07, Loss_ic: 7.85446e-07, Loss_b: 3.36106e-09
Iter 39100, Loss: 1.00507e-06, Loss_res: 2.16274e-07, Loss_ic: 7.85434e-07, Loss_b: 3.36044e-09
Iter 39200, Loss: 1.00504e-06, Loss_res: 2.16259e-07, Loss_ic: 7.85422e-07, Loss_b: 3.35983e-09
Iter 39300, Loss: 1.00501e-06, Loss_res: 2.16244e-07, Loss_ic: 7.85410e-07, Loss_b: 3.35923e-09
Iter 39400, Loss: 1.00499e-06, Loss_res: 2.16230e-07, Loss_ic: 7.85399e-07, Loss_b: 3.35862e-09
Iter 39500, Loss: 1.00496e-06, Loss_res: 2.16215e-07, Loss_ic: 7.85387e-07, Loss_b: 3.35801e-09
Iter 396

In [11]:
all_states1 = {"model": model.net.state_dict(), "LBFGS": model.optimizer2.state_dict()}
torch.save(obj=all_states1, f="/kaggle/working/all_states1.pth")

In [12]:
q_pred, r_pred = model.predict(X)

In [13]:
H_pred = griddata(X, q_pred.flatten(), (x_mesh,t_mesh), method='cubic')
H_True = griddata(X, r_pred.flatten(), (x_mesh,t_mesh), method='cubic')

In [ ]:
def plot_all_3d(T, X, H_True, H_pred, smooth_factor=1.0, dpi=150):
    """
    绘制对比3D图像（真实值、预测值和差值），并优化显示效果
    
    参数:
    - smooth_factor: 控制插值平滑程度的因子
    - dpi: 图像分辨率
    """
    # 创建更高分辨率的图形
    fig = plt.figure(figsize=(21, 5), dpi=dpi)
    
    # 定义共享的颜色映射和视角
    cmap = cm.get_cmap('rainbow')  # 
    #cmap = cm.get_cmap('viridis')  # 使用更现代的颜色映射
    elev, azim = 30, 45  # 定义共享的视角
    
    # 创建更平滑的网格数据（可选）
    if smooth_factor > 1:
        from scipy.ndimage import gaussian_filter
        H_True_smooth = gaussian_filter(H_True, sigma=smooth_factor)
        H_pred_smooth = gaussian_filter(H_pred, sigma=smooth_factor)
    else:
        H_True_smooth = H_True
        H_pred_smooth = H_pred
    
    # 计算误差
    error = np.abs(H_True - H_pred)
    
    # 定义绘图函数，减少代码重复
    def plot_surface(ax, X, Y, Z, title, cmap=cmap):
        surf = ax.plot_surface(X, Y, Z, cmap=cmap, 
                              linewidth=0, antialiased=True,
                              rstride=1, cstride=1,
                              alpha=0.95)
        ax.set_xlabel('$Tv$')
        ax.set_ylabel('$Z$')
        ax.set_zlabel('h')
        ax.set_title(title)
        ax.view_init(elev=elev, azim=azim)
        fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.05)
        # 添加网格线增强深度感
        ax.grid(True, linestyle='-', alpha=0.3)
        return surf
    
    # 真实值3D图
    ax0 = fig.add_subplot(131, projection='3d')
    plot_surface(ax0, T, X, H_True_smooth, 'Real')
    
    # 预测值3D图
    ax1 = fig.add_subplot(132, projection='3d')
    plot_surface(ax1, T, X, H_pred_smooth, 'Prediction')
    
    # 差距3D图
    ax2 = fig.add_subplot(133, projection='3d')
    # 使用专门的误差颜色映射
    #error_cmap = cm.get_cmap('Reds')
    plot_surface(ax2, T, X, error, 'Gap')
    
    # 调整布局，确保有足够的空间显示标签和颜色条
    plt.tight_layout(pad=3.0)
    plt.show()

# 示例调用
plot_all_3d(x_mesh, t_mesh, H_True, H_pred, smooth_factor=1.2, dpi=200)

In [15]:
data = {
    'x': x_mesh.flatten()[:, None],
    't': t_mesh.flatten()[:, None],
    'q_pred': q_pred.flatten()[:, None],
    'r_pred': r_pred.flatten()[:, None]
}

scipy.io.savemat('/kaggle/working/1all_data.mat', data)

In [16]:
loss_total = model.loss_history['total']
loss_residual = model.loss_history['residual']
loss_ic = model.loss_history['ic']
loss_b = model.loss_history['b']

In [17]:
num_iterations = len(loss_total)  # 损失记录的总数量（即总迭代次数）
iter_steps = list(range(100, 100 + num_iterations * 100, 100))
# 3. 将数据转换为竖排（列向量）：通过reshape(-1, 1)实现，形状变为(n, 1)
iter_steps_col = np.array(iter_steps).reshape(-1, 1)       # 竖排迭代步数
loss_total_col = np.array(loss_total).reshape(-1, 1)       # 竖排总损失
loss_residual_col = np.array(loss_residual).reshape(-1, 1) # 竖排残差损失
loss_ic_col = np.array(loss_ic).reshape(-1, 1)             # 竖排初始条件损失
loss_b_col = np.array(loss_b).reshape(-1, 1)               # 竖排边界条件损失

# 4. 整理并保存为mat文件（此时所有数据均为列向量）
data_loss = {
    'iter_steps': iter_steps_col,
    'loss_total': loss_total_col,
    'loss_residual': loss_residual_col,
    'loss_ic': loss_ic_col,
    'loss_b': loss_b_col
}

scipy.io.savemat('/kaggle/working/1loss_history.mat', data_loss)